In [ ]:
#pip install langchain

In [ ]:
#pip install rapidocr-onnxruntime

In [ ]:
#pip install langchain_community

In [ ]:
#pip install langchain_huggingface

In [ ]:
#pip install langchain_ollama

In [ ]:
#pip install pypdf

In [ ]:
import pdb

In [ ]:
#pip install faiss-cpu

In [ ]:
import os
from langchain.chains import create_retrieval_chain
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain


In [ ]:
def load_documents():
    root_folder = "/Users/harinivaranasi/Desktop/Research"
    
    loader = DirectoryLoader(
        path=root_folder,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        loader_kwargs={"extract_images": True},
        recursive=True   
    )
    
    documents = loader.load()
    breakpoint()
    return documents

In [ ]:
load_documents()

In [ ]:
def create_retriever(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=200,
        separators=["\n\n", "\n", " "]
    )
    texts = text_splitter.split_documents(documents)
    
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-mpnet-base-v2',
        model_kwargs={"device": "cpu"}
    )
    
    db = FAISS.from_documents(texts, embeddings)
    db.save_local("faiss_index")
    
    return db.as_retriever(search_type='mmr', search_kwargs={"k": 5})

In [ ]:
def load_faiss_index():
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-mpnet-base-v2',
        model_kwargs={"device": "cpu"}
    )
    
    return FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True).as_retriever(
        search_type='mmr', search_kwargs={"k": 5}
    )

In [ ]:
'''def build_retrieval_qa_chain(retriever):
    
   # system_prompt = (
    #    "You are a highly knowledgeable environmental scientist."
     #   " Use the provided context to answer questions factually and concisely."
      #  "The responses should be concise and precise."
       # " If the answer cannot be derived from the context, respond with"
        #" 'The information is not available in the provided context.'"
        #" Do not add any external information or assumptions."
        #" Context: {context}"
   # )
        system_prompt = (
        "You are a highly knowledgeable environmental scientist. "
        "Use the provided context to answer the following four questions factually and concisely. "
        "Respond only using information found in the context. Do not speculate. "
        "If the answer cannot be found in the context, use 'Not available'.\n\n"
        "Questions:\n"
        "1. What is the name of the data source used in the research?\n"
        "2. What are the variables from this source used in the research?\n"
        "3. What is the time period covered by the data?\n"
        "4. What is the geographical location or coverage of the data?\n\n"
        "Provide your answer in the following structured format:\n"
        "{\n"
        "  \"data_source\": <name>,\n"
        "  \"variables\": [<var1>, <var2>, ...],\n"
        "  \"time_period\": <time range>,\n"
        "  \"location\": <region>\n"
        "}\n\n"
        "Context: {context}"
    )
        

    model = OllamaLLM(
        model='llama3.2',
        temperature=0.3,
        max_tokens=512,
        top_p=0.85
    )
    
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}"),
        ]
    )

    stuff_chain = create_stuff_documents_chain(model, prompt)

    retrieval_chain = create_retrieval_chain(
        retriever,
        stuff_chain
    )

    return retrieval_chain'''

In [ ]:
def build_retrieval_qa_chain_v2(retriever):
    system_prompt = (
        "You are a highly knowledgeable environmental scientist. "
        "Use the provided context to answer the following four questions factually and concisely. "
        "Respond only using information found in the context. Do not speculate. "
        "If the answer cannot be found in the context, use 'Not available'.\n\n"
        "Context: {context}"
    )

    model = OllamaLLM(
        model='llama3.2',
        temperature=0.3,
        max_tokens=512,
        top_p=0.85
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{input}")
        ]
    )

    stuff_chain = create_stuff_documents_chain(model, prompt)

    retrieval_chain = create_retrieval_chain(
        retriever,
        stuff_chain
    )

    return retrieval_chain


In [ ]:
def build_summarization_chain(model):
    """
    Builds a summarization chain using the provided model.

    Args:
        model: An instance of the language model to perform summarization.

    Returns:
        A summarization chain ready to process input texts.
    """
   
    # Define the summarization prompt
    system_prompt = (
        "You are a summarization assistant."
        " Summarize the following text into a concise and coherent summary,"
        " highlighting the most important points. Avoid unnecessary details."
        " Text to summarize: {text}"
    )

    # Create a chat prompt template
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            ("human", "{text}"),
        ]
    )

    # Create the summarization chain
    try:
        summarization_chain = LLMChain(
            llm=model,
            prompt=summarization_prompt
        )
    except Exception as e:
        raise ValueError("Failed to create the summarization chain. Ensure the model is correctly configured.") from e

    return summarization_chain

In [ ]:
def create_retriever():
    documents = load_documents()  # Load fresh documents
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-mpnet-base-v2',
        model_kwargs={"device": "cpu"}
    )
    
    retriever = FAISS.from_documents(documents, embeddings).as_retriever(
        search_type='mmr', search_kwargs={"k": 3}
    )
    
    retriever.vectorstore.save_local("faiss_index")  # Save new FAISS index
    return retriever

In [ ]:
def classify_query(query):
    """
    Classifies whether the query is for summarization or QA.
    Args:
        query (str): The user's input query.

    Returns:
        str: "summarization" or "qa" based on the query type.
    """
    keywords = ["summarize", "overview", "brief", "condense"]  # Add more if needed
    for keyword in keywords:
        if keyword in query.lower():
            return "summarization"
    return "qa"

def main():
    # Load or create the retriever
    #retriever = load_faiss_index() if os.path.exists("faiss_index") else create_retriever(load_documents())
    if os.path.exists("faiss_index"):
        os.system("rm -r faiss_index")  # Remove the previous FAISS index

    retriever = create_retriever()
    # Build the QA and summarization chains
    qa_chain = build_retrieval_qa_chain_v2(retriever)
    model = OllamaLLM(model='llama3.2', temperature=0.3, max_tokens=256, top_p=0.9)
    summarization_chain = build_summarization_chain(model)

    while True:
        user_query = input("Type your query here (or type 'exit' to quit): ")

        if user_query.lower() == 'exit':
            print("Exiting the chatbot. Goodbye!")
            break

        if user_query:
            # Classify the query
            query_type = classify_query(user_query)

            try:
                if query_type == "summarization":
                    # Handle summarization
                    documents = retriever.get_relevant_documents(user_query)  # Retrieve relevant document(s)
                    if documents:
                        text_to_summarize = documents[0].page_content  # Get the main content
                        summary = summarization_chain.run({"text": text_to_summarize})
                        print("Summary:", summary)
                    else:
                        print("No relevant documents found to summarize.")

                elif query_type == "qa":
                    # Handle QA
                    response = qa_chain.invoke({"input": user_query})

                    # Print the answer
                    if 'answer' in response:
                        print("Response:", response['answer'])

                    # Print source documents
                    if 'context' in response:
                        print("\nSource Documents:")
                        for doc in response['context']:
                            print(f" - Source: {doc.metadata['source']}, Page: {doc.metadata.get('page', 'N/A')}")
                else:
                    print("Unable to classify the query.")

            except Exception as e:
                print("An error occurred while processing your query:", str(e))
        else:
            print("Please enter a valid query.")

if __name__ == "__main__":
    main()
